0. 라이브러리 설치

In [ ]:
# YOLO 모델 학습 및 추론을 위한 Ultralytics 설치
# Roboflow 데이터셋 다운로드를 위한 roboflow 패키지 설치
!pip install ultralytics roboflow

1. 라이브러리 import

In [ ]:
# YOLO 모델 학습, 검증, 예측, 변환에 사용
from ultralytics import YOLO

# Roboflow 데이터셋 다운로드에 사용
from roboflow import Roboflow

# Colab에서 파일 업로드 및 다운로드에 사용
from google.colab import files
from google.colab import userdata

# 파일 및 경로 처리를 위해 사용
import os
import glob

2. Roboflow 데이터셋 다운로드

Roboflow에서 생성한 `bus_door` 데이터셋을 Colab으로 불러온다.  
API Key는 GitHub에 노출되지 않도록 Colab Secrets를 사용한다.

Colab Secrets 등록:
- Name: `ROBOFLOW_API_KEY`
- Value: 본인 Roboflow API Key

In [ ]:
# Colab Secrets에 저장된 Roboflow API Key를 불러온다.
# GitHub에 API Key가 노출되지 않도록 코드에 직접 작성하지 않는다.
ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

# Roboflow 객체 생성
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Roboflow workspace와 project 지정
project = rf.workspace("oseongs-workspace-lmlnu").project("bus_door_detection")

# 사용할 데이터셋 버전 선택
version = project.version(5)

# YOLOv8 형식으로 데이터셋 다운로드
dataset = version.download("yolov8")

3. 데이터셋 구조 확인

In [ ]:
# 다운로드된 데이터셋 위치 확인
print("Dataset location:", dataset.location)

# 데이터셋 폴더 내부 확인
!ls {dataset.location}

4. COCO pretrained YOLO 모델 불러오기

빈 모델을 처음부터 학습시키는 것이 아니라, COCO 데이터셋으로 사전학습된 YOLO 모델을 불러와 `bus_door` 데이터로 파인튜닝한다.

`yolov8n.pt`는 YOLOv8 nano 모델로, 비교적 가벼워 모바일 적용 가능성을 검토하기에 적합하다.

In [ ]:
# COCO 데이터셋으로 사전학습된 YOLOv8n 모델 불러오기
model = YOLO("yolov8n.pt")

5. bus_door 데이터로 파인튜닝

In [ ]:
# Roboflow에서 다운로드한 data.yaml을 이용해 bus_door 데이터셋으로 모델을 학습한다.
# data: 학습 데이터셋 설정 파일
# epochs: 학습 반복 횟수
# imgsz: 입력 이미지 크기
# batch: 한 번에 학습할 이미지 수
# name: 학습 결과 저장 폴더 이름

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="bus_door_yolo"
)

6. 학습된 모델 불러오기

In [ ]:
# 학습 결과 중 가장 성능이 좋은 best.pt 모델을 불러온다.
best_model = YOLO("runs/detect/bus_door_yolo/weights/best.pt")

# best.pt 파일 위치 확인
!find runs/detect -name best.pt

7. 모델 성능 검증

In [ ]:
# validation 데이터셋을 기준으로 모델 성능을 평가한다.
metrics = best_model.val(
    data=f"{dataset.location}/data.yaml",
    imgsz=640
)

# mAP50: IoU 0.5 기준 평균 정밀도
print("mAP50:", metrics.box.map50)

# mAP50-95: IoU 0.5~0.95 범위 기준 평균 정밀도
print("mAP50-95:", metrics.box.map)

# Precision: 모델이 탐지했다고 한 것 중 실제로 맞은 비율
print("Precision:", metrics.box.mp)

# Recall: 실제 객체 중 모델이 놓치지 않고 잡은 비율
print("Recall:", metrics.box.mr)

8. 검증 이미지 예측

In [ ]:
# validation 이미지 전체에 대해 예측을 수행한다.
# conf=0.5 이상인 탐지만 결과로 저장한다.
best_model.predict(
    source=f"{dataset.location}/valid/images",
    conf=0.5,
    save=True,
    name="valid_prediction_test"
)

9. 새로운 이미지 테스트

In [ ]:
# 학습에 사용하지 않은 새로운 버스 이미지를 업로드한다.
uploaded = files.upload()

# 업로드된 파일명 가져오기
filename = list(uploaded.keys())[0]

# 새로운 이미지에 대해 bus_door 탐지 수행
results = best_model.predict(
    source=filename,
    conf=0.5,
    save=True,
    name="new_image_test"
)

10. 여러 장 이미지 테스트

In [ ]:
# 새로운 테스트 이미지를 저장할 폴더 생성
os.makedirs("new_test_images", exist_ok=True)

# 여러 장의 이미지 업로드
uploaded = files.upload()

# 업로드된 이미지를 new_test_images 폴더로 이동
for filename in uploaded.keys():
    os.rename(filename, f"new_test_images/{filename}")

# 폴더 전체 이미지에 대해 예측 수행
results = best_model.predict(
    source="new_test_images",
    conf=0.5,
    save=True,
    name="new_bus_door_test"
)

11. 영상 테스트

In [ ]:
# 버스 영상 파일 업로드
uploaded = files.upload()

# 업로드된 영상 파일명 가져오기
video_name = list(uploaded.keys())[0]

# 영상 프레임별 bus_door 탐지 수행
results = best_model.predict(
    source=video_name,
    conf=0.5,
    save=True,
    name="bus_door_video_test"
)

12. TFLite 변환

In [ ]:
# 학습된 best.pt 모델을 모바일 환경에서 사용할 수 있는 TFLite 형식으로 변환한다.
export_path = best_model.export(
    format="tflite",
    imgsz=640
)

print("Exported model path:", export_path)

14. TFLite 파일 확인

In [ ]:
# 생성된 TFLite 파일 경로 확인
!find . -name "*.tflite"

15. TFLite 모델 테스트

In [ ]:
# 변환된 TFLite 모델 불러오기
# 아래 경로는 find 결과에 맞게 수정해야 한다.
tflite_model = YOLO("./runs/detect/bus_door_yolo/weights/best_saved_model/best_float32.tflite")


# 새로운 이미지 업로드
uploaded = files.upload()

# 업로드된 이미지 파일명 가져오기
filename = list(uploaded.keys())[0]

# TFLite 모델로 예측 수행
results = tflite_model.predict(
    source=filename,
    conf=0.5,
    save=True,
    name="tflite_single_test"
)

15. TFLite 파일 다운로드

In [ ]:
# 변환된 TFLite 모델 파일 다운로드
files.download("./runs/detect/bus_door_yolo/weights/best_saved_model/best_float32.tflite")